In [ ]:
#!pip install alpaca-py #newer base
#!pip install alpaca-trade-api
#!pip install mplfinance

In [3]:
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from mplfinance.original_flavor import candlestick_ohlc

In [4]:
# Alpaca API credentials
API_KEY = "PKP6G2PCSLR7KABZBU9Z"
API_SECRET = "mPNf9KoilrmoMdz4MedQdeaZWxda3D1YjgCIJbjd"

client = StockHistoricalDataClient(API_KEY, API_SECRET)

In [5]:
def get_stock_data(symbol: str, start_date: str, end_date: str):
    request_params = StockBarsRequest(
        symbol_or_symbols=[symbol],
        timeframe=TimeFrame.Day,
        start=datetime.strptime(start_date, "%Y-%m-%d"),
        end=datetime.strptime(end_date, "%Y-%m-%d")
    )

    bars = client.get_stock_bars(request_params)
    df = bars.df

    # Filter columns to include only OHLC and volume
    return df[['open', 'high', 'low', 'close', 'volume']]

#df = get_stock_data("AAPL", "2024-05-01", "2024-05-10")
#print(df)

In [6]:
def plot_candlestick_with_volume(df, symbol: str):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True, 
                                   gridspec_kw={'height_ratios': [3, 1]})

    # Prepare data for candlestick
    quotes = []
    for idx, (multi_idx, row) in enumerate(df.iterrows()):
        # multi_idx is a tuple: (symbol, timestamp)
        timestamp = multi_idx[1]
        quotes.append((
            mdates.date2num(timestamp),
            row['open'],
            row['high'],
            row['low'],
            row['close']
        ))

    # Candlestick chart
    candlestick_ohlc(ax1, quotes, width=0.6, colorup='g', colordown='r', alpha=0.8)
    ax1.set_title(f'{symbol} Candlestick Chart')
    ax1.set_ylabel('Price')
    ax1.grid(True)

    # Volume bar chart
    ax2.bar([multi_idx[1] for multi_idx in df.index], df['volume'], color='gray', width=0.6)
    ax2.set_ylabel('Volume')
    ax2.grid(True)

    # Format x-axis with dates
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


In [8]:
# Example usage
symbol = "AAPL"
start_date = "2024-05-10"
end_date = "2024-05-10"
df = get_stock_data(symbol, start_date, end_date)
plot_candlestick_with_volume(df, symbol)

KeyError: "None of [Index(['open', 'high', 'low', 'close', 'volume'], dtype='object')] are in the [columns]"